In [29]:
import json
import os
import sys
from prettytable import PrettyTable

In [30]:
RESULTS_DIR = "../results/"

ALGORITHMS = [
    "ippo",
    "mappo",
    "pimac_v0",
    "pimac_v6"
]

NAME_MAP = {
    "ippo": "IPPO",
    "mappo": "MAPPO",
    "pimac_v0": "PIC-MAPPO",
    "pimac_v6": "TC3D"
}

TASKS = [
    "spread",
    "rware",
    "lbf_hard"
]

SEEDS = [42, 43, 44, 45, 46]

tc3d_selections = {
    "rware" : "active_01",
    "spread" : "active_03",
    "lbf_hard" : "active_01"
}

In [31]:
def get_path(algorithm, task, seed, tc3d_selection=None):
    dir_path = os.path.join(RESULTS_DIR, task, algorithm)
    # all files in directory
    files = os.listdir(dir_path)
    # find the filename which includes the seed in its name
    files = [f for f in files if f"s{seed}" in f]
    if tc3d_selection is not None:
        files = [f for f in files if tc3d_selection in f]
    if len(files) == 0:
        raise ValueError(f"No file found for algorithm {algorithm}, task {task}, seed {seed}: files={files}")
    if len(files) > 1:
        raise ValueError(f"Multiple files found for algorithm {algorithm}, task {task}, seed {seed}: files={files}")
    return os.path.join(dir_path, files[0])

In [32]:
def make_table(data, task):
    table_top = """
    \\begin{table}[t]
    \\centering
    \\caption{{task_name}}
    \\label{{tab:{task_name}}}
    \\begin{tabular}{@{}lccc@{}}
    \\toprule
    Method & Train & Validation & Test \\\\ \\midrule
    """

    table_bottom = """  
    \\bottomrule
    \\end{tabular}
    \\end{table}
    """
    if task == "lbf_hard":
        table_top = table_top.replace("{task_name}", f"{task.replace("_", "\\_")} ($\\times 1e2$)")
    else:
        table_top = table_top.replace("{task_name}", task.replace("_", "\\_"))
    rows = []
    for algorithm in data:
        row = f"{NAME_MAP[algorithm]} & {data[algorithm]['train']} & {data[algorithm]['val']} & {data[algorithm]['test']} \\\\"
        rows.append(row)
    return table_top + "\n".join(rows) + table_bottom
    

In [33]:
all_scores = {}
for task in TASKS:
    print("--------------------------------------------------")
    print(f"Task: {task}")
    #print("--------------------------------------------------")
    task_scores = {}
    for algorithm in ALGORITHMS:
        #print(f"  Algorithm: {algorithm}")
        scores = {"train": [], "val": [], "test": []}
        for seed in SEEDS:
            if algorithm == "pimac_v6":
                path = get_path(algorithm, task, seed, tc3d_selection=tc3d_selections[task])
            else:
                path = get_path(algorithm, task, seed)
            summary_json = os.path.join(path, "summary.json")
            with open(summary_json, "r") as f:
                data = json.load(f)
                train_score = data["test"]["train_counts_mean"]
                val_score = data["test"]["validation_counts_mean"]
                test_score = data["test"]["test_counts_mean"]
                scores["train"].append(train_score)
                scores["val"].append(val_score)
                scores["test"].append(test_score)
        for key in scores:
            mean_score = sum(scores[key]) / len(scores[key])
            std_score = (sum((x - mean_score) ** 2 for x in scores[key]) / (len(scores[key]) - 1)) ** 0.5
            #scores[key] = (mean_score, std_score)
            if task == "lbf_hard":
                mean_score *= 100
                std_score *= 100
            scores[key] = f"${round(mean_score, 2)}$ $\\pm$ ${round(std_score, 1)}$"
        task_scores[algorithm] = scores
    print(make_table(task_scores, task))
    table = PrettyTable()
    table.field_names = ["Method", "Train", "Validation", "Test"]
    for algorithm in ALGORITHMS:
        scores = task_scores[algorithm]
        row = [NAME_MAP[algorithm]]
        for key in ["train", "val", "test"]:
            row.append(scores[key].replace("$", "").replace("\\pm", "±"))
        table.add_row(row)
    #print(table)
    #print("--------------------------------------------------")
    all_scores[task] = task_scores

--------------------------------------------------
Task: spread

    \begin{table}[t]
    \centering
    \caption{spread}
    \label{{tab:spread}}
    \begin{tabular}{@{}lccc@{}}
    \toprule
    Method & Train & Validation & Test \\ \midrule
    IPPO & $-57.06$ $\pm$ $1.6$ & $-65.01$ $\pm$ $2.0$ & $-103.43$ $\pm$ $2.6$ \\
MAPPO & $-42.45$ $\pm$ $0.8$ & $-51.96$ $\pm$ $1.3$ & $-86.13$ $\pm$ $1.7$ \\
PIC-MAPPO & $-42.0$ $\pm$ $0.5$ & $-50.34$ $\pm$ $0.9$ & $-84.42$ $\pm$ $1.3$ \\
TC3D & $-39.9$ $\pm$ $0.7$ & $-48.09$ $\pm$ $1.0$ & $-79.18$ $\pm$ $1.5$ \\  
    \bottomrule
    \end{tabular}
    \end{table}
    
--------------------------------------------------
Task: rware

    \begin{table}[t]
    \centering
    \caption{rware}
    \label{{tab:rware}}
    \begin{tabular}{@{}lccc@{}}
    \toprule
    Method & Train & Validation & Test \\ \midrule
    IPPO & $2.58$ $\pm$ $1.5$ & $2.58$ $\pm$ $1.5$ & $6.17$ $\pm$ $3.0$ \\
MAPPO & $1.07$ $\pm$ $0.6$ & $0.99$ $\pm$ $0.6$ & $2.77$ $\pm$ $1.6$

In [34]:
def big_table(all_scores):
    table_top = """
    \\begin{table}[t]
    \\centering
    \\caption{All results}
    \\label{tab:all_results}
    \\resizebox{\\textwidth}{!}{%
    \\begin{tabular}{@{}lccc|ccc|ccc@{}}
    \\multirow{2}{*}{Method} & \\multicolumn{3}{c}{Spread} & \\multicolumn{3}{c}{LBF ($\\times 1e2$)}   & \\multicolumn{3}{c}{RWARE} \\\\
        & Train  & Validation & Test & Train & Validation & Test & Train & Validation & Test \\\\ \\midrule
    """
    
    table_bottom = """ \\bottomrule
    \\end{tabular}
    }
    \\end{table}
    """
    rows = []
    for algorithm in ALGORITHMS:
        row = f"{NAME_MAP[algorithm]} & \
            {all_scores['spread'][algorithm]['train']} & {all_scores['spread'][algorithm]['val']} & {all_scores['spread'][algorithm]['test']} & \
                {all_scores['lbf_hard'][algorithm]['train']} & {all_scores['lbf_hard'][algorithm]['val']} & {all_scores['lbf_hard'][algorithm]['test']} & \
                {all_scores['rware'][algorithm]['train']} & {all_scores['rware'][algorithm]['val']} & {all_scores['rware'][algorithm]['test']} \\\\"
        rows.append(row)
    return table_top + "\n".join(rows) + table_bottom

In [35]:
print(big_table(all_scores))


    \begin{table}[t]
    \centering
    \caption{All results}
    \label{tab:all_results}
    \resizebox{\textwidth}{!}{%
    \begin{tabular}{@{}lccc|ccc|ccc@{}}
    \multirow{2}{*}{Method} & \multicolumn{3}{c}{Spread} & \multicolumn{3}{c}{LBF ($\times 1e2$)}   & \multicolumn{3}{c}{RWARE} \\
        & Train  & Validation & Test & Train & Validation & Test & Train & Validation & Test \\ \midrule
    IPPO &             $-57.06$ $\pm$ $1.6$ & $-65.01$ $\pm$ $2.0$ & $-103.43$ $\pm$ $2.6$ &                 $6.84$ $\pm$ $1.4$ & $3.98$ $\pm$ $0.7$ & $8.24$ $\pm$ $0.6$ &                 $2.58$ $\pm$ $1.5$ & $2.58$ $\pm$ $1.5$ & $6.17$ $\pm$ $3.0$ \\
MAPPO &             $-42.45$ $\pm$ $0.8$ & $-51.96$ $\pm$ $1.3$ & $-86.13$ $\pm$ $1.7$ &                 $5.61$ $\pm$ $0.6$ & $3.44$ $\pm$ $0.5$ & $7.91$ $\pm$ $0.8$ &                 $1.07$ $\pm$ $0.6$ & $0.99$ $\pm$ $0.6$ & $2.77$ $\pm$ $1.6$ \\
PIC-MAPPO &             $-42.0$ $\pm$ $0.5$ & $-50.34$ $\pm$ $0.9$ & $-84.42$ $\pm$ $1.3$ &          